In [1]:
import pandas as pd

In [2]:
DATA_PATH = "../data/raw/en_fr.parquet"

In [3]:
df = pd.read_parquet(DATA_PATH)
df.head()

,id,translation
0,0,"{'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}"
1,1,"{'en': 'Alain-Fournier', 'fr': 'Alain-Fournier'}"
2,2,"{'en': 'First Part', 'fr': 'PREMIÈRE PARTIE'}"
3,3,"{'en': 'I', 'fr': 'CHAPITRE PREMIER'}"
4,4,"{'en': 'THE BOARDER', 'fr': 'LE PENSIONNAIRE'}"


In [4]:
df["en"] = df["translation"].apply(lambda x: x["en"])
df["fr"] = df["translation"].apply(lambda x: x["fr"])

df = df[["en", "fr"]]
df.head()

,en,fr
0,The Wanderer,Le grand Meaulnes
1,Alain-Fournier,Alain-Fournier
2,First Part,PREMIÈRE PARTIE
3,I,CHAPITRE PREMIER
4,THE BOARDER,LE PENSIONNAIRE


## Préparation

Le nettoyage, la suppression des doublons et le découpage sont reproductibles dans `scripts/prepare_data.py`. Ce notebook reste réservé à l'exploration.

In [8]:
train_df = pd.read_parquet("../data/raw/train.parquet")
train_df.head()

,en,fr
0,"The truth is, he ought to have been trusted wi...","La vérité est que j'eusse dû lui confier tout,..."
1,"To absorb it, we would need to fill containers...","Pour l'absorber, il eût fallu remplir des réci..."
2,"""Ah, that’s quite another thing; but promise m...",-- Alors c'est autre chose; mais promettez-moi...
3,"""Now for the lad; look sharp.""","Au mioche maintenant, dépechons!"
4,'I would give my life a thousand times to know...,– Je donnerais mille fois ma vie pour savoir c...


In [9]:
from machine_translation import (
    get_data_loaders,
    load_config,
    load_tokenizer,
)

In [10]:
CONFIG_PATH = "../configs/seq2seq_tatoeba.yaml"
config = load_config(CONFIG_PATH)
tokenizer_config = config.tokenizer
data_config = config.data

In [11]:
source_tokenizer = load_tokenizer(
    tokenizer_config.source_artifact_path, tokenizer_config.special_tokens
)
target_tokenizer = load_tokenizer(
    tokenizer_config.target_artifact_path, tokenizer_config.special_tokens
)

In [12]:
sample = "Before we start, let's make sure we have the right tools for the job."

encoded_sample = source_tokenizer.encode(sample)
print("Encoded sample:", encoded_sample)
print("Encoded sample:", encoded_sample.ids)
print("Decoded sample:", source_tokenizer.decode(encoded_sample.ids))
print("Tokens:", encoded_sample.tokens)

Encoded sample: Encoding(num_tokens=19, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
Encoded sample: [2, 3283, 59, 1831, 4, 287, 60, 163, 511, 59, 37, 5, 343, 4455, 24, 5, 8318, 6, 3]
Decoded sample: Before we start, let's make sure we have the right tools for the job.
Tokens: ['<bos>', '▁Before', '▁we', '▁start', ',', '▁let', "'s", '▁make', '▁sure', '▁we', '▁have', '▁the', '▁right', '▁tools', '▁for', '▁the', '▁job', '.', '<eos>']


In [13]:
train_loader, validation_loader, test_loader = get_data_loaders(
    source_tokenizer=source_tokenizer,
    target_tokenizer=target_tokenizer,
    data_config=data_config,
    special_tokens=tokenizer_config.special_tokens,
)

In [14]:
len(train_loader), len(validation_loader), len(test_loader)

(1589, 199, 199)

In [16]:
for batch in validation_loader:
    print(batch["source_ids"])
    print(batch["target_ids"])
    print(batch["source_padding_mask"])
    print(batch["target_padding_mask"])
    print("Source input IDs shape:", batch["source_ids"].shape)
    print("Target input IDs shape:", batch["target_ids"].shape)
    print("Source padding mask shape:", batch["source_padding_mask"].shape)
    print("Target padding mask shape:", batch["target_padding_mask"].shape)
    break

tensor([[   2,   71, 9107,  ...,    0,    0,    0],
        [   2,  467, 6603,  ...,    0,    0,    0],
        [   2,   45,   98,  ...,    0,    0,    0],
        ...,
        [   2,  290,  237,  ...,    0,    0,    0],
        [   2, 3611,   10,  ...,    0,    0,    0],
        [   2,   11,   13,  ...,    0,    0,    0]])
tensor([[    2,    50,  8834,  ...,     0,     0,     0],
        [    2,    73,   444,  ...,     0,     0,     0],
        [    2,   155,    79,  ...,     0,     0,     0],
        ...,
        [    2,    12, 11528,  ...,     0,     0,     0],
        [    2, 13320,  2060,  ...,     0,     0,     0],
        [    2,  1206,    18,  ...,     0,     0,     0]])
tensor([[False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [

<pad>
